In [12]:
import sys
!{sys.executable} -m pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.12.0
Uninstalling torch-2.12.0:
  Successfully uninstalled torch-2.12.0
Found existing installation: torchvision 0.27.0
Uninstalling torchvision-0.27.0:
  Successfully uninstalled torchvision-0.27.0
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


You can safely remove it manually.
You can safely remove it manually.


In [1]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --no-cache-dir --prefer-binary

^C


In [14]:
# Importando as bibliotecas

import torch
import torch.nn as nn
import torch.nn.functional as F 
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [15]:
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.optim as optim

# Configura as transformações para a ResNet (Imagens maiores)
transformacao_resnet = transforms.Compose([
    transforms.Resize(224), # Estica as imagens do CIFAR-10
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# Carrega os dados de TREINO com a nova transformação
dataset_treino = torchvision.datasets.CIFAR10(root='./dados', train=True, download=True, transform=transformacao_resnet)
dataloader_treino = DataLoader(dataset_treino, batch_size=32, shuffle=True)

# Prepara o dispositivo (Usa Placa de Vídeo se tiver, senão usa Processador)
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinando na unidade: {dispositivo}")

# Baixa a ResNet18 e adapta a última camada para 10 classes
modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
numero_de_recursos = modelo.fc.in_features
modelo.fc = nn.Linear(numero_de_recursos, 10)
modelo = modelo.to(dispositivo) # Envia o modelo para a GPU/CPU

# Define o Criterio e o Otimizador
criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo.parameters(), lr=0.001)

C:\Users\Windows\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Treinando na unidade: cpu


In [7]:
from torch.utils.data import Dataset
import numpy as np

# Função fornecida no site do Dataset
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dicionario = pickle.load(fo, encoding='bytes')
    return dicionario

class DatasetCIFAR(Dataset):
    def __init__(self, file_path, transform=None):
        # Carrega o dicionário
        dicionario = unpickle(file_path)
        
        # Guarda as imagens na chave data e os rótulos em labels
        self.dados = dicionario[b'data']
        self.rotulos = dicionario[b'labels']
        self.transform = transform
        
        # N = número de imagens, 3 = canais RGB, 32x32 = tamanho da imagem
        self.dados = self.dados.reshape(-1, 3, 32, 32)

    def __len__(self):
        # Retorna a quantidade total de imagens no arquivo
        return len(self.dados)

    def __getitem__(self, idx):
        # Pega uma imagem e seu rótulo específico
        imagem = self.dados[idx]
        rotulo = self.rotulos[idx]
        
        # Imagens em array no formato (Altura, Largura, Canais) para aplicar as transformações do PyTorch
        # Trocamos a ordem de (3, 32, 32) para (32, 32, 3) temporariamente para aplicar as transformações 
        imagem = np.transpose(imagem, (1, 2, 0))
        
        if self.transform:
            imagem = self.transform(imagem)
            
        return imagem, rotulo

In [8]:
import torchvision.models as models
import torch.nn as nn

# Baixa o modelo ResNet18, usei o 18 ao invés do 50 para ser mais leve, mas o processo é o mesmo
modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Modifica apenas a camada de saída
# A ResNet original tenta classificar 1000 objetos diferentes, mas só temos 10
# Então, pegamos o número de neurônios que chegam na última camada (in_features) e trocamos a última camada ("fc" = fully connected) por uma nossa, com 10 saídas.
numero_de_recursos = modelo.fc.in_features
modelo.fc = nn.Linear(numero_de_recursos, 10)

# Manda o modelo para a Placa de Vídeo se houver
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo = modelo.to(dispositivo)

In [9]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Configura o transformer para normalizar as imagens
transformacao = transforms.Compose([
    transforms.Resize(224), # Estica a imagem de 32x32 para 224x224
    transforms.ToTensor(),
    # Normaliza as imagens com a média e desvio padrão usados no ImageNet
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# Carrega os dados (lembre de colocar o caminho certo do seu arquivo!)
caminho_do_arquivo = 'cifar-10-batches-py/test_batch' 
dataset_teste = DatasetCIFAR(caminho_do_arquivo, transform=transformacao)
dataloader_teste = DataLoader(dataset_teste, batch_size=32, shuffle=False)

C:\Users\Windows\AppData\Local\Temp\ipykernel_20044\1154434655.py:8: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dicionario = pickle.load(fo, encoding='bytes')


In [11]:
epocas = 1 # 1 época de treino, optei por só um porque 5 estava demorando demais, mas o processo é o mesmo para mais épocas

print("Iniciando o treinamento...")

for epoca in range(epocas):
    modelo.train() 
    perda_acumulada = 0.0
    
    for imagens, rotulos in dataloader_treino:

        # Manda as imagens e rótulos para a Placa de Vídeo se houver, não foi preciso no CNN padrão pois é um modelo leve, mas a ResNet é pesada e é bom ter uma GPU para treinar em um tempo razoável
        imagens = imagens.to(dispositivo)
        rotulos = rotulos.to(dispositivo)
        
        # Limpar a memória do aprendizado anterior
        otimizador.zero_grad()
        
        # Forward (A rede tenta adivinhar o que é a imagem)
        saidas = modelo(imagens)
        
        # Calcular o erro 
        erro = criterio(saidas, rotulos)
        
        # Backward 
        erro.backward()
        
        # Atualiza os pesos
        otimizador.step()
        
        perda_acumulada += erro.item()
        
    # Mostra o resultado ao final de cada época
    erro_medio = perda_acumulada / len(dataloader_treino)
    print(f"Época {epoca+1}/{epocas} concluída! Erro médio: {erro_medio:.4f}")

print("Treinamento finalizado!")

Iniciando o treinamento...


KeyboardInterrupt: 